In [2]:
"""
Design
- FD grid is used as a coordinate scaffold only. Candidate points are
  drawn from FD grid coordinates so coverage is always dense.
- u values at interior points are computed by PDE-informed interpolation,
  NOT copied from FD. FD is used only for error measurement.
- Stopping criterion: max_points successfully computed interior points.
- KDTree for O(log N) neighbour queries.
- O(1) candidate removal via swap-and-pop.
"""

import time
import numpy as np
from scipy.spatial import KDTree
from scipy.interpolate import RegularGridInterpolator

alpha = 0.05
c     = 0.5
T     = 0.5
import numpy as np

alpha = 0.05
c = 0.5

beta = c / (2 * alpha)
_N_SERIES = 200

# Precompute coefficients once
_xq = np.linspace(0.0, 1.0, 20001)
_b = np.zeros(_N_SERIES + 1)

for n in range(1, _N_SERIES + 1):
    integrand = np.exp(-beta * _xq) * np.sin(np.pi * _xq) * np.sin(n * np.pi * _xq)
    _b[n] = 2.0 * np.trapezoid(integrand, _xq)

_n_arr = np.arange(1, _N_SERIES + 1)
_bn    = _b[1:]
_lam   = -alpha * (_n_arr * np.pi) ** 2

def u_true(x, t):
    x = np.asarray(x, dtype=float)
    scalar = x.ndim == 0
    x = np.atleast_1d(x)
    sin_nx = np.sin(np.pi * np.outer(x, _n_arr))
    exp_t  = np.exp(_lam * t)
    s      = (sin_nx * (_bn * exp_t)).sum(axis=-1)
    result = np.exp(beta * x - (c**2 / (4 * alpha)) * t) * s
    return float(result[0]) if scalar else result

# ── FD r/eference grid ─────────────────────────────────────────────────────────
nx_fd = 100
x_fd  = np.linspace(0.0, 1.0, nx_fd)
dx_fd = x_fd[1] - x_fd[0]

# Compute nt_fd from CFL — never touch this manually
dt_cfl = min(0.9  * dx_fd / c,
             0.45 * dx_fd**2 / alpha)
nt_fd  = max(int(np.ceil(T / dt_cfl)) + 1, 10)
t_fd   = np.linspace(0.0, T, nt_fd)
dt_fd  = t_fd[1] - t_fd[0]



def build_fd_reference():
    """Explicit upwind FD solve. Returns (u_fd, interpolant)."""
    u = np.zeros((nx_fd, nt_fd))
    u[:, 0] = u_true(x_fd, 0.0)

    r_diff = alpha * dt_fd / dx_fd**2
    r_adv  = c     * dt_fd / dx_fd

    for it in range(nt_fd - 1):
        un = u[:, it]
        u[1:-1, it+1] = (un[1:-1]
                         - r_adv  * (un[1:-1] - un[:-2])
                         + r_diff * (un[2:] - 2*un[1:-1] + un[:-2]))
        u[0,  it+1] = 0.0
        u[-1, it+1] = 0.0

    interp = RegularGridInterpolator(
        (x_fd, t_fd), u, method='linear',
        bounds_error=False, fill_value=None
    )
    return u, interp


# ── Integrator hyperparameters ────────────────────────────────────────────────
max_points    = 100000   # stopping criterion
u_bound = 1.1
n_neighbours  = 5     # stencil size (3 equations, 5 unknowns → underdetermined)
cond_max      = 1e4
isotropy_max  = 50.0
w_max_factor  = 3.0     # reject if max|w_j| > w_max_factor / n
w_min_floor   = -1.0

# Spawn box in FD grid steps
spawn_nx      = 5       # ± x steps around a newly solved point
spawn_nt      = 3       # + t steps forward

# Neighbour search box in normalised units (x/dx_fd, t/dt_fd)
box_n0        = 2.0     # initial half-width
box_grow      = 1.5
box_nmax      = 15.0

max_candidates = 6000   # cap on candidate list length (for memory and speed)


# ── KDTree-backed visited set ─────────────────────────────────────────────────
class VisitedSet:
    """
    Stores (x, t, u) points. KDTree is in normalised (x/dx_fd, t/dt_fd)
    space so both axes are commensurate during neighbour search.
    Rebuilt lazily every rebuild_every insertions.
    """
    def __init__(self, rebuild_every=200):
        self._pts           = []
        self._norm          = []
        self._tree          = None
        self._dirty         = 0
        self._rebuild_every = rebuild_every
    def add(self, x, t, u):
        self._pts.append((x, t, u))
        self._norm.append((x / dx_fd, t / dt_fd))   # add this
        self._dirty += 1
        if self._dirty >= self._rebuild_every:
            self._rebuild()

    def _rebuild(self):
        self._tree  = KDTree(np.array(self._norm))   # change this
        self._dirty = 0

    def _ensure(self):
        if self._tree is None:
            self._rebuild()

    def find_neighbours(self, x_star, t_star):
        self._ensure()
        xn = x_star / dx_fd
        tn = t_star / dt_fd
        bn = box_n0
        while True:
            idxs = self._tree.query_ball_point([xn, tn], r=bn, p=np.inf)
            nb = [
                self._pts[i] for i in idxs
                if abs(self._pts[i][0] - x_star) <= bn * dx_fd
                and 0 < (t_star - self._pts[i][1]) <= bn * dt_fd
            ]
            if len(nb) >= n_neighbours:
                nb.sort(key=lambda p: (p[0]-x_star)**2 + (p[1]-t_star)**2)
                return nb[:n_neighbours]
            bn = min(bn * box_grow, box_nmax)
            if bn >= box_nmax:
                return None
    def __len__(self):
        return len(self._pts)


# ── Stencil geometry check ────────────────────────────────────────────────────
def stencil_is_good(dx_i, dt_i):
    if np.min(dx_i) * np.max(dx_i) > 0:          # two-sided coverage
        return False
    h   = max(np.max(np.abs(dx_i)), 1e-12)
    tau = max(np.max(np.abs(dt_i)), 1e-12)
    pts = np.column_stack([dx_i / h, dt_i / tau])
    return True


# ── Weight solver ─────────────────────────────────────────────────────────────
def solve_weights(dx_i, dt_i):
    """
    Min-norm lstsq on the 3×n PDE-informed system.
    Returns (w, cond) or (None, cond).
    """
    h   = max(np.max(np.abs(dx_i)), 1e-12)
    tau = max(np.max(np.abs(dt_i)), 1e-12)

    A = np.array([
        np.ones(len(dx_i)),
         (dx_i - c * dt_i)              / h,
        (0.5 * dx_i**2 + alpha * dt_i) / h**2,
    ]) 
    b = np.array([1 , 0.0, 0.0])

    cond = np.linalg.cond(A)
    if cond > cond_max:
        return None,    cond

    w = np.linalg.lstsq(A, b, rcond=None)[0]


    # Additional weight checks to prevent error explosion — these are somewhat ad-hoc and may be tuned or removed based on your needs.
    if np.max(np.abs(w)) > w_max_factor / len(dx_i):   # no dominant weight
        return None, cond
    if np.min(w) < w_min_floor:                         # no extreme negatives
        return None, cond

    return w, cond

## Add L2 instead of L1

# ── Main integrator ───────────────────────────────────────────────────────────
def run_integrator(seed=None):
    t_wall = time.time()
    if seed is not None:
        np.random.seed(seed)

    print("Building FD reference...", flush=True)
    _, fd_interp = build_fd_reference()
    print(f"  dx={dx_fd:.4f}  dt={dt_fd:.5f}", flush=True)

    # IC + BC into visited set
    visited = VisitedSet()
    for ix in range(nx_fd):
        visited.add(x_fd[ix], 0.0, u_true(x_fd[ix], 0.0))
    for it in range(1, nt_fd):
        visited.add(0.0,  t_fd[it], 0.0)
        visited.add(1.0,  t_fd[it], 0.0)
    print(f"  IC+BC points: {len(visited)}", flush=True)

    # Candidate pool — integer (ix, it) keys for O(1) deduplication
    cand_keys = set()
    cand_list = []   # list of (x, t)

    def push(ix, it):
        if (ix, it) not in cand_keys:
            cand_keys.add((ix, it))
            cand_list.append((x_fd[ix], t_fd[it]))

    # Seed from IC
    for ix in range(1, nx_fd - 1):
        for dix in range(-spawn_nx, spawn_nx + 1):
            for dit in range(1, spawn_nt + 1):
                nix, nit = ix + dix, dit
                if 1 <= nix <= nx_fd - 2 and nit < nt_fd:
                    push(nix, nit)

    print(f"  Initial candidates: {len(cand_list)}", flush=True)
    print(f"  Stopping at {max_points} computed points\n", flush=True)

    rej        = dict(proximity=0, no_nb=0, geometry=0, cond=0, weights=0)
    point_data = []

    while cand_list and len(point_data) < max_points:

        # Trim if candidate list is over budget
        protect_candidates = 500

        if len(cand_list) > max_candidates:
            target_size = max(max_candidates, protect_candidates)
            n_remove = len(cand_list) - target_size
            if n_remove > 0:
                del cand_list[:n_remove]

        # ── O(1) swap-and-pop ─────────────────────────────────────────────
        idx            = np.random.randint(len(cand_list))
        x_star, t_star = cand_list[idx]
        cand_list[idx] = cand_list[-1]
        cand_list.pop()
        ix_s = int(round(x_star / dx_fd))
        it_s = int(round(t_star / dt_fd))
        cand_keys.discard((ix_s, it_s))
        

        nb = visited.find_neighbours(x_star, t_star)
        if nb is None:
            rej['no_nb'] += 1
            continue

        nb_pts = np.array(nb)
        dx_i   = nb_pts[:, 0] - x_star
        dt_i   = nb_pts[:, 1] - t_star   # negative: neighbours are in the past

        if not stencil_is_good(dx_i, dt_i):
            rej['geometry'] += 1
            continue

        w, cond = solve_weights(dx_i, dt_i)
        if w is None:
            rej['cond' if cond > cond_max else 'weights'] += 1
            continue

        # Accept point  
        u_hat = float(w @ nb_pts[:, 2])

        # Break the error cascade — u is bounded by IC amplitude + margin
        if abs(u_hat) > u_bound:
            rej['bounds'] = rej.get('bounds', 0) + 1
            continue

        visited.add(x_star, t_star, u_hat)

        # Spawn new candidates in the FD grid box around this point
        for dix in range(-spawn_nx, spawn_nx + 1):
            for dit in range(1, spawn_nt + 1):
                nix = ix_s + dix
                nit = it_s + dit
                if 1 <= nix <= nx_fd - 2 and nit < nt_fd:
                    push(nix, nit)

        u_fd_ref = float(fd_interp([[x_star, t_star]])[0])
        u_true_ref = float(u_true(x_star, t_star))
 
        point_data.append({
            'x':     x_star,
            't':     t_star,
            'u':     u_hat,
            'u_true': u_true_ref,
            'u_fd':   u_fd_ref,
            'err_true': abs(u_hat - u_true_ref),
            'err_fd':   abs(u_hat - u_fd_ref),
            'err':   abs(u_hat - u_true_ref),   # use true error for stats and plotting
            'cond':  cond,
            'w1':    float(np.sum(np.abs(w))),
            'box_x': float(np.max(np.abs(dx_i))),   # ← add this
            'box_t': float(np.max(np.abs(dt_i))),   # ← add this
            'nb_x':  nb_pts[:, 0].copy(),            # ← add
            'nb_t':  nb_pts[:, 1].copy(),            # ← add
            'nb_u':  nb_pts[:, 2].copy(),            # ← add
        })

        if len(point_data) % 500 == 0:
            errs = [p['err'] for p in point_data]
            t_max = max(p['t'] for p in point_data)
            print(f"  {len(point_data):5d} pts | "
                  f"mean={np.mean(errs):.2e} | "
                  f"max={np.max(errs):.2e} | "
                  f"t_max={t_max:.3f} | "
                  f"cands={len(cand_list):5d} | "
                  f"wall={time.time()-t_wall:.1f}s",
                  flush=True)

    # Summary
    errs = np.array([p['err'] for p in point_data])
    print(f"\n{'─'*56}")
    print(f"  Computed  : {len(point_data)} points")
    print(f"  Mean err  : {errs.mean():.3e}")
    print(f"  Max  err  : {errs.max():.3e}")
    print(f"  Wall time : {time.time()-t_wall:.1f}s")
    total = sum(rej.values()) + len(point_data)
    print(f"\n  Rejections (total attempts: {total})")
    for r, n in rej.items():
        print(f"    {r:10s}: {n:6d}  ({100*n/total:.1f}%)")

    return visited, point_data, fd_interp


#if __name__ == '__main__':
 #   visited, point_data, fd_interp = run_integrator()

In [4]:
"""
GFDMEnv — Gymnasium environment for RL-based point placement.

The agent controls N particles that start at the initial condition (t=0)
and must navigate to a fixed target point (x*, t*) in the space-time domain.
At each step every particle simultaneously outputs a displacement
(delta_x, delta_t) drawn from a discrete set of FD grid multiples.
The GFDM solver is called at each new particle position to compute the
solution value and stencil quality metrics, which drive the reward signal.

Observation space : Box(5N,)  — [x_i, t_i, u_i, cond_i, w1_i] per particle,
                                  all normalised to roughly [-1, 1] or [0, 1].
Action space      : MultiDiscrete([N_DX * N_DT] * N)
                    — one flattened (dx_idx, dt_idx) per particle.
Reward            : dense step penalty on stencil quality + speed,
                    sparse terminal reward on prediction error at z*.
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces

 
# ── Discrete action grid ───────────────────────────────────────────────────────
# dx choices: -4 … +4 grid steps  (9 options)
# dt choices:  0 … +4 grid steps  (5 options, 0 = stay still)
DX_STEPS = np.array([-4, -3, -2, -1, 0, 1, 2, 3, 4], dtype=int)   # 9 options
DT_STEPS = np.array([0, 1, 2, 3, 4],                  dtype=int)   # 5 options
N_DX     = len(DX_STEPS)   # 9
N_DT     = len(DT_STEPS)   # 5
N_ACTS   = N_DX * N_DT     # 45 actions per particle


def _decode_action(action_idx: int):
    """Map a flat action index in [0, 44] → (dx_steps, dt_steps)."""
    dx_idx = action_idx // N_DT
    dt_idx = action_idx  % N_DT
    return DX_STEPS[dx_idx], DT_STEPS[dt_idx]


# ── Normalisation constants ───────────────────────────────────────────────────
_NORM_X    = 1.0        # x already in [0, 1]
_NORM_T    = T          # t in [0, T]
_NORM_U    = u_bound    # u clipped to [-u_bound, u_bound]
_NORM_COND = cond_max   # cond in [1, cond_max]
_NORM_W1   = 5.0        # w1 is typically O(1); cap at 5 for safety


class GFDMEnv(gym.Env):
    """
    Parameters
    ----------
    N          : number of particles (= number of IC points used)
    target_ix  : x-index of the target point on the FD grid (int)
    target_it  : t-index of the target point on the FD grid (int)
    lam        : weight on cond(A) in the dense reward
    mu         : weight on delta_t in the dense reward (speed penalty)
    r_reject   : penalty assigned when GFDM solve is rejected
    c_arrive   : bonus added to the terminal reward upon reaching z*
    K_max      : maximum number of steps before episode is truncated
    seed       : optional numpy random seed
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        N          : int   = 10,
        target_ix  : int   = 50,   # x* = x_fd[50] = 0.505…
        target_it  : int   = 50,   # t* = t_fd[50]
        lam        : float = 0.1,
        mu         : float = 1.0,
        r_reject   : float = -2.0,
        c_arrive   : float = 5.0,
        K_max      : int   = 200,
        seed       : int   = None,
    ):
        super().__init__()

        # ── Validate target ───────────────────────────────────────────────
        assert 1 <= target_ix <= nx_fd - 2, "target_ix must be an interior x-index"
        assert 1 <= target_it <= nt_fd - 1, "target_it must be a positive t-index"

        self.N         = N
        self.target_ix = target_ix
        self.target_it = target_it
        self.x_star    = x_fd[target_ix]
        self.t_star    = t_fd[target_it]
        self.lam       = lam
        self.mu        = mu
        self.r_reject  = r_reject
        self.c_arrive  = c_arrive
        self.K_max     = K_max
        self._rng      = np.random.default_rng(seed)

        # N evenly spaced IC indices (interior points only)
        self._ic_indices = np.linspace(1, nx_fd - 2, N, dtype=int)

        # ── Gymnasium spaces ──────────────────────────────────────────────
        # Observation: 5 features per particle, normalised to [-1, 1] / [0, 1]
        self.observation_space = spaces.Box(
            low  = -1.0,
            high =  1.0,
            shape= (5 * N,),
            dtype= np.float32,
        )

        # Action: one flat integer in [0, N_ACTS) per particle
        self.action_space = spaces.MultiDiscrete([N_ACTS] * N)

        # ── Internal state (reset on each episode) ────────────────────────
        self._visited    = None   # VisitedSet
        self._pos_ix     = None   # shape (N,) integer x-indices
        self._pos_it     = None   # shape (N,) integer t-indices
        self._u          = None   # shape (N,) solution values at current positions
        self._cond       = None   # shape (N,) last cond(A) per particle
        self._w1         = None   # shape (N,) last ||w||_1 per particle
        self._step_count = 0

    # ──────────────────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Re-seed if requested
        if seed is not None:
            self._rng = np.random.default_rng(seed)

        # Build fresh visited set with IC + BC
        self._visited = VisitedSet()
        for ix in range(nx_fd):
            self._visited.add(x_fd[ix], 0.0, float(u_true(x_fd[ix], 0.0)))
        for it in range(1, nt_fd):
            self._visited.add(0.0, t_fd[it], 0.0)
            self._visited.add(1.0, t_fd[it], 0.0)

        # Place particles at IC positions
        self._pos_ix = self._ic_indices.copy()
        self._pos_it = np.zeros(self.N, dtype=int)   # all at t = 0

        # Solution values at IC (known exactly)
        self._u = np.array([
            float(u_true(x_fd[ix], 0.0)) for ix in self._pos_ix
        ], dtype=float)

        # Stencil quality initialised to neutral values
        self._cond = np.ones(self.N,  dtype=float)
        self._w1   = np.ones(self.N,  dtype=float)

        self._step_count = 0

        return self._get_obs(), {}

    # ──────────────────────────────────────────────────────────────────────────
    def step(self, action):
        """
        action : np.ndarray of shape (N,), each element in [0, N_ACTS).
        """
        assert len(action) == self.N

        total_reward = 0.0
        done         = False
        info         = {"accepted": [], "rejected": [], "terminal": False}

        for i, act in enumerate(action):
            ddx, ddt = _decode_action(int(act))

            # Proposed new grid indices
            new_ix = int(self._pos_ix[i]) + ddx
            new_it = int(self._pos_it[i]) + ddt

            # ── Hard domain constraints ───────────────────────────────────
            # Must stay in interior x and not go backward in t
            if not (1 <= new_ix <= nx_fd - 2):
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue
            if not (0 <= new_it <= nt_fd - 1):
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue
            if ddt < 0:
                # Causality: backward moves not yet supported
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue

            x_new = x_fd[new_ix]
            t_new = t_fd[new_it]

            # ── If particle is at t=0 (IC), value is known exactly ────────
            if new_it == 0:
                u_new  = float(u_true(x_new, 0.0))
                cond_i = 1.0
                w1_i   = 1.0
                self._pos_ix[i] = new_ix
                self._pos_it[i] = new_it
                self._u[i]      = u_new
                self._cond[i]   = cond_i
                self._w1[i]     = w1_i
                self._visited.add(x_new, t_new, u_new)
                # Small penalty for not moving forward in time
                total_reward += -self.mu * ddt
                info["accepted"].append(i)
                continue

            # ── GFDM solve ────────────────────────────────────────────────
            nb = self._visited.find_neighbours(x_new, t_new)
            if nb is None:
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue

            nb_pts = np.array(nb)
            dx_i   = nb_pts[:, 0] - x_new
            dt_i   = nb_pts[:, 1] - t_new

            if not stencil_is_good(dx_i, dt_i):
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue

            w, cond_i = solve_weights(dx_i, dt_i)
            if w is None:
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue

            u_new = float(w @ nb_pts[:, 2])

            # Bounds check
            if abs(u_new) > u_bound:
                total_reward += self.r_reject
                info["rejected"].append(i)
                continue

            w1_i = float(np.sum(np.abs(w)))

            # ── Accept: update particle state and visited list ────────────
            self._pos_ix[i] = new_ix
            self._pos_it[i] = new_it
            self._u[i]      = u_new
            self._cond[i]   = cond_i
            self._w1[i]     = w1_i
            self._visited.add(x_new, t_new, u_new)
            info["accepted"].append(i)

            # Dense reward: stencil quality + speed penalty
            total_reward += -(w1_i + self.lam * cond_i + self.mu * ddt * dt_fd)

            # ── Check terminal condition ──────────────────────────────────
            if new_ix == self.target_ix and new_it == self.target_it:
                u_true_val = float(u_true(self.x_star, self.t_star))
                error      = abs(u_new - u_true_val)
                total_reward += -error + self.c_arrive
                done                = True
                info["terminal"]    = True
                info["final_error"] = error
                break   # episode ends; ignore remaining particles

        self._step_count += 1
        truncated = (not done) and (self._step_count >= self.K_max)

        return self._get_obs(), float(total_reward), done, truncated, info

    # ──────────────────────────────────────────────────────────────────────────
    def _get_obs(self) -> np.ndarray:
        """
        Returns the normalised state vector of shape (5N,).
        Layout per particle: [x_norm, t_norm, u_norm, cond_norm, w1_norm]
        """
        obs = np.empty(5 * self.N, dtype=np.float32)
        for i in range(self.N):
            base = 5 * i
            obs[base + 0] = x_fd[self._pos_ix[i]] / _NORM_X
            obs[base + 1] = t_fd[self._pos_it[i]] / _NORM_T
            obs[base + 2] = np.clip(self._u[i]    / _NORM_U,    -1.0, 1.0)
            obs[base + 3] = np.clip(self._cond[i] / _NORM_COND,  0.0, 1.0)
            obs[base + 4] = np.clip(self._w1[i]   / _NORM_W1,    0.0, 1.0)
        return obs

    # ──────────────────────────────────────────────────────────────────────────
    def render(self):
        """Print a simple text summary of current particle positions."""
        print(f"\n--- Step {self._step_count} ---")
        print(f"  Target : x*={self.x_star:.4f}  t*={self.t_star:.4f}")
        for i in range(self.N):
            x_i = x_fd[self._pos_ix[i]]
            t_i = t_fd[self._pos_it[i]]
            print(f"  P{i:02d}: x={x_i:.4f}  t={t_i:.4f}  "
                  f"u={self._u[i]:.4f}  "
                  f"cond={self._cond[i]:.1f}  "
                  f"w1={self._w1[i]:.3f}")
        print(f"  Visited points: {len(self._visited)}")


# ── Sanity check ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Building environment...")
    env = GFDMEnv(N=10, target_ix=50, target_it=50, K_max=100)

    obs, _ = env.reset(seed=42)
    print(f"Observation shape : {obs.shape}")
    print(f"Action space      : {env.action_space}")
    print(f"Target            : x*={env.x_star:.4f}  t*={env.t_star:.4f}")

    print("\nRunning 20 random steps...")
    total_r  = 0.0
    for step in range(2000):
        action = env.action_space.sample()
        obs, reward, done, truncated, info = env.step(action)
        total_r += reward
        print(f"  step={step+1:02d}  reward={reward:+.4f}  "
              f"accepted={info['accepted']}  "
              f"terminal={info['terminal']}")
        if done or truncated:
            print("  Episode finished.")
            break

    print(f"\nTotal reward over {step+1} steps: {total_r:.4f}")
    env.render()

Building environment...
Observation shape : (50,)
Action space      : MultiDiscrete([45 45 45 45 45 45 45 45 45 45])
Target            : x*=0.5051  t*=0.0459

Running 20 random steps...
  step=01  reward=-12.6339  accepted=[1, 2, 3, 4, 5, 6, 7, 8, 9]  terminal=False
  step=02  reward=-16.5297  accepted=[1, 5, 8, 9]  terminal=False
  step=03  reward=-18.6503  accepted=[2, 5, 6, 7]  terminal=False
  step=04  reward=-17.6699  accepted=[1, 4, 8, 9]  terminal=False
  step=05  reward=-18.7835  accepted=[3, 5, 6]  terminal=False
  step=06  reward=-19.0404  accepted=[0, 2, 4, 8, 9]  terminal=False
  step=07  reward=-19.0577  accepted=[2, 4, 9]  terminal=False
  step=08  reward=-20.3657  accepted=[0, 2, 9]  terminal=False
  step=09  reward=-19.7495  accepted=[0, 2, 7]  terminal=False
  step=10  reward=-19.5579  accepted=[3]  terminal=False
  step=11  reward=-18.2334  accepted=[6, 7, 8, 9]  terminal=False
  step=12  reward=-19.5579  accepted=[3]  terminal=False
  step=13  reward=-18.6737  accept

In [6]:
env = GFDMEnv(N=5, target_ix=50, target_it=50, K_max=100)
obs, _ = env.reset(seed=42)

for step in range(500):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    print(f"step={step+1:02d}  reward={reward:+.4f}  accepted={info['accepted']}  rejected={info['rejected']}")
    if done or truncated:
        print(f"Episode ended at step {step+1}")
        break

step=01  reward=-8.5240  accepted=[1, 2, 3]  rejected=[0, 4]
step=02  reward=-8.9681  accepted=[0, 2]  rejected=[1, 3, 4]
step=03  reward=-8.2697  accepted=[0, 1, 2, 4]  rejected=[3]
step=04  reward=-9.0475  accepted=[2, 4]  rejected=[0, 1, 3]
step=05  reward=-10.1873  accepted=[2]  rejected=[0, 1, 3, 4]
step=06  reward=-9.4859  accepted=[4]  rejected=[0, 1, 2, 3]
step=07  reward=-9.3086  accepted=[1, 3]  rejected=[0, 2, 4]
step=08  reward=-10.0000  accepted=[]  rejected=[0, 1, 2, 3, 4]
step=09  reward=-10.1238  accepted=[0]  rejected=[1, 2, 3, 4]
step=10  reward=-9.4375  accepted=[1]  rejected=[0, 2, 3, 4]
step=11  reward=-9.5588  accepted=[4]  rejected=[0, 1, 2, 3]
step=12  reward=-9.3051  accepted=[0, 1, 3]  rejected=[2, 4]
step=13  reward=-9.1955  accepted=[3, 4]  rejected=[0, 1, 2]
step=14  reward=-9.5579  accepted=[3]  rejected=[0, 1, 2, 4]
step=15  reward=-10.0000  accepted=[]  rejected=[0, 1, 2, 3, 4]
step=16  reward=-10.0000  accepted=[]  rejected=[0, 1, 2, 3, 4]
step=17  rewa

In [6]:
# How often does each particle get accepted vs rejected?
n_episodes = 20
accept_counts = np.zeros(10)
reject_counts = np.zeros(10)

for _ in range(n_episodes):
    obs, _ = env.reset()
    for _ in range(100):
        action = env.action_space.sample()
        obs, reward, done, truncated, info = env.step(action)
        for i in info['accepted']: accept_counts[i] += 1
        for i in info['rejected']: reject_counts[i] += 1
        if done or truncated:
            break

for i in range(10):
    total = accept_counts[i] + reject_counts[i]
    print(f"P{i:02d}: accept={accept_counts[i]/total*100:.1f}%  reject={reject_counts[i]/total*100:.1f}%")

P00: accept=10.4%  reject=89.6%
P01: accept=10.8%  reject=89.1%
P02: accept=12.8%  reject=87.2%
P03: accept=11.8%  reject=88.2%
P04: accept=12.2%  reject=87.8%
P05: accept=13.8%  reject=86.2%
P06: accept=13.4%  reject=86.7%
P07: accept=12.4%  reject=87.5%
P08: accept=11.4%  reject=88.6%
P09: accept=12.3%  reject=87.6%
